# Corrected $\psi$–$\theta$ and PDG comparison

This version uses the **recorded GasBox entry and exit boundaries**. It does not use the last non-zero energy-deposit hit as the exit, and all positions remain in Geant4 mm.

- $\theta_x=\mathrm{atan2}(p_x,p_z)$ is the signed projected exit-direction angle.
- $\psi_x=\mathrm{atan2}(x_{out}-x_{in},z_{out}-z_{in})$ is the signed endpoint-displacement angle.
- Full RMS is used for the $\theta/\psi=\sqrt{3}$ transport check.
- Central-98% Gaussian-equivalent width is used for the PDG core-width comparison.

In [ ]:
import math
import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

file_path = '/Users/s.x./Data/TPC_Simulation/entry_scan_mu_p1000_L100_lateral_100k.root'
particle_mass_MeV = 105.6583755
momentum_MeV_c = 1000.0
gas_length_cm = 100.0

In [ ]:
branches = [
    'Event_no', 'KE', 'entry_KE', 'entered',
    'entry_pos_x', 'entry_pos_y', 'entry_pos_z',
    'exit_pos_x', 'exit_pos_y', 'exit_pos_z',
    'exit_momentum_x', 'exit_momentum_y', 'exit_momentum_z',
]
with uproot.open(file_path) as root_file:
    df = root_file['Exit'].arrays(branches, library='pd').sort_values('Event_no')

# Keep particles that entered the gas and reached the downstream face.
expected_exit_z_mm = 5.0 * gas_length_cm
df = df[(df['entered'] == 1) & np.isclose(df['exit_pos_z'], expected_exit_z_mm, atol=1e-6)].copy()
print(f'{len(df):,} particles reached the downstream face')
print(f"entry z = {df['entry_pos_z'].mean():.1f} mm, exit z = {df['exit_pos_z'].mean():.1f} mm")

In [ ]:
# Direct projected angles; no spherical-coordinate rotation or 3D/sqrt(2) conversion is needed.
for axis in ('x', 'y'):
    df[f'theta_{axis}'] = np.arctan2(df[f'exit_momentum_{axis}'], df['exit_momentum_z'])
    df[f'psi_{axis}'] = np.arctan2(
        df[f'exit_pos_{axis}'] - df[f'entry_pos_{axis}'],
        df['exit_pos_z'] - df['entry_pos_z'],
    )

def full_rms(values):
    return np.std(np.asarray(values), ddof=0)

def central98_gaussian_sigma(values):
    values = np.asarray(values)
    lo, hi = np.quantile(values, [0.01, 0.99])
    central = values[(values >= lo) & (values <= hi)]
    # A central-98% truncated unit Gaussian has standard deviation 0.934553.
    return np.std(central, ddof=0) / 0.934553

rows = []
for axis in ('x', 'y'):
    theta_rms = full_rms(df[f'theta_{axis}'])
    psi_rms = full_rms(df[f'psi_{axis}'])
    rows.append({
        'projection': axis,
        'theta_full_RMS_deg': np.degrees(theta_rms),
        'psi_full_RMS_deg': np.degrees(psi_rms),
        'theta_over_psi_full_RMS': theta_rms / psi_rms,
        'theta_core_sigma_deg': np.degrees(central98_gaussian_sigma(df[f'theta_{axis}'])),
    })
transport = pd.DataFrame(rows)
transport

In [ ]:
# PDG/Highland projected core width for the same particle, momentum, gas and length.
density_g_cm3 = 0.00665333
X0_g_cm2 = 1.0 / (0.175/94.32 + 0.225/42.70 + 0.600/34.24)
beta = momentum_MeV_c / np.sqrt(momentum_MeV_c**2 + particle_mass_MeV**2)
x_over_X0 = gas_length_cm * density_g_cm3 / X0_g_cm2
sigma_PDG_rad = (13.6 / (beta * momentum_MeV_c)) * np.sqrt(x_over_X0) * (1 + 0.038*np.log(x_over_X0/beta**2))
sigma_PDG_deg = np.degrees(sigma_PDG_rad)
transport['sigma_PDG_deg'] = sigma_PDG_deg
transport['Geant4_core_over_PDG'] = transport['theta_core_sigma_deg'] / sigma_PDG_deg
print(f'sigma_PDG = {sigma_PDG_deg:.5f} deg')
print(f"mean full-RMS theta/psi = {transport['theta_over_psi_full_RMS'].mean():.4f}; sqrt(3) = {np.sqrt(3):.4f}")
transport

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, axis in zip(axes, ('x', 'y')):
    ax.hist(np.degrees(df[f'theta_{axis}']), bins=140, density=True, histtype='step', label=rf'$\theta_{axis}$')
    ax.hist(np.degrees(df[f'psi_{axis}']), bins=140, density=True, histtype='step', label=rf'$\psi_{axis}$')
    ax.set(xlabel='projected angle [deg]', ylabel='normalized entries', title=f'{axis} projection')
    ax.legend()
plt.tight_layout()

## Interpretation

Do not compare the central-98% $\theta$ width to the central-98% $\psi$ width and demand exactly $\sqrt{3}$. The textbook $\sqrt{3}$ relation is a second-moment (full-RMS) result for the same transport distribution. The PDG comparison instead concerns the approximately Gaussian projected angular core.